# Notebook 6 – Interaction Features

## What are Interaction Features?

Interaction features are new features created by combining two or more existing features.

They help represent relationships between features that may not be captured when each feature is considered independently.

Common interaction methods include:

- Feature × Feature
- Feature + Feature
- Feature − Feature
- Feature / Feature
- Polynomial Features
- Pairwise Interactions
- Domain-Based Interactions

### Examples

- Income × Age
- Quantity × Price
- Experience / Age
- Total Spending / Number of Orders

### When are Interaction Features Useful?

Interaction features are useful when the effect of one feature depends on another feature.

For example, in the Titanic dataset, the effect of passenger class may be related to age. Combining `Age` and `Pclass` can provide the model with additional information.

### When Can They Create Unnecessary Complexity?

Creating too many interactions can:

- Increase the number of features
- Increase model complexity
- Increase computational cost
- Create redundant features
- Increase the risk of overfitting
- Make the model harder to interpret

Therefore, interaction features should be created based on domain knowledge or a clear reason rather than creating every possible combination.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Titanic-Dataset.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Understanding the Source Features

The Titanic dataset contains numerical features such as:

- `Age`
- `SibSp`
- `Parch`
- `Fare`
- `Pclass`

These features can be combined to create meaningful interaction features.

For this notebook, the interactions will be selected based on their potential meaning rather than creating every possible combination.

In [2]:
df[["Age", "SibSp", "Parch", "Fare", "Pclass"]].head()

,Age,SibSp,Parch,Fare,Pclass
0,22.0,1,0,7.2500,3
1,38.0,1,0,71.2833,1
2,26.0,0,0,7.9250,3
3,35.0,1,0,53.1000,1
4,35.0,0,0,8.0500,3


## 1. Feature × Feature

Multiplication combines two features by multiplying their values.

### Feature Name
`Age_Pclass_Interaction`

### Source Columns
- `Age`
- `Pclass`

### Logic
`Age × Pclass`

### Reason
The relationship between passenger age and passenger class may contain useful information.

### Business Meaning
In a general business dataset, multiplication can represent combined effects such as income × age or quantity × price.

### ML Relevance
The interaction can help a model represent relationships between two features.

### Leakage Check
Neither feature uses the target variable, so there is no direct target leakage.

### Final Decision
**Retain**

In [3]:
df["Age_Pclass_Interaction"] = df["Age"] * df["Pclass"]

df[["Age", "Pclass", "Age_Pclass_Interaction"]].head()

,Age,Pclass,Age_Pclass_Interaction
0,22.0,3,66.0
1,38.0,1,38.0
2,26.0,3,78.0
3,35.0,1,35.0
4,35.0,3,105.0


## 2. Feature + Feature

Addition combines two features to represent their total.

### Feature Name
`Family_Total`

### Source Columns
- `SibSp`
- `Parch`

### Logic
`SibSp + Parch`

### Reason
It represents the number of relatives traveling with the passenger.

### Business Meaning
In a business dataset, addition can combine related quantities into a total.

### ML Relevance
The combined value may provide more useful information than considering the two family relationship features independently.

### Leakage Check
The target variable is not used.

### Final Decision
**Retain**

In [4]:
df["Family_Total"] = df["SibSp"] + df["Parch"]

df[["SibSp", "Parch", "Family_Total"]].head()

,SibSp,Parch,Family_Total
0,1,0,1
1,1,0,1
2,0,0,0
3,1,0,1
4,0,0,0


## 3. Feature − Feature

Subtraction measures the difference between two features.

### Feature Name
`Sibling_Parent_Difference`

### Source Columns
- `SibSp`
- `Parch`

### Logic
`SibSp − Parch`

### Reason
It identifies the difference between siblings/spouses and parents/children traveling with the passenger.

### Business Meaning
In business data, subtraction can represent differences such as income − expenses or current value − previous value.

### ML Relevance
Difference features can capture relative relationships that individual features may not represent.

### Leakage Check
The target variable is not used.

### Final Decision
**Retain**

In [5]:
df["Sibling_Parent_Difference"] = df["SibSp"] - df["Parch"]

df[["SibSp", "Parch", "Sibling_Parent_Difference"]].head()

,SibSp,Parch,Sibling_Parent_Difference
0,1,0,1
1,1,0,1
2,0,0,0
3,1,0,1
4,0,0,0


## 4. Feature / Feature

Division creates a ratio between two features.

### Feature Name
`Fare_Per_Family_Member`

### Source Columns
- `Fare`
- `SibSp`
- `Parch`

### Logic
`Fare / (SibSp + Parch + 1)`

The `+1` includes the passenger in the family group and prevents division by zero.

### Reason
The feature estimates the fare associated with each person in the passenger's family group.

### Business Meaning
A similar business feature could be total spending / number of orders.

### ML Relevance
Ratios can represent relative behavior and may be more informative than absolute values.

### Leakage Check
The target is not used. In a real ML workflow, the calculation must still be performed consistently on training and test data.

### Final Decision
**Retain**

In [6]:
family_size = df["SibSp"] + df["Parch"] + 1

df["Fare_Per_Family_Member"] = df["Fare"] / family_size

df[["Fare", "SibSp", "Parch", "Fare_Per_Family_Member"]].head()

,Fare,SibSp,Parch,Fare_Per_Family_Member
0,7.2500,1,0,3.62500
1,71.2833,1,0,35.64165
2,7.9250,0,0,7.92500
3,53.1000,1,0,26.55000
4,8.0500,0,0,8.05000


## 5. Polynomial Features

Polynomial features create higher-order terms from existing numerical features.

For example:

- `Age`
- `Age²`
- `Age³`

They can help a model capture non-linear relationships.

### Feature Name
`Age_Squared`

### Source Columns
- `Age`

### Logic
`Age²`

### Reason
The effect of age on the target may not always be linear.

### Business Meaning
A polynomial transformation can represent situations where the effect of a variable changes as its value increases.

### ML Relevance
Polynomial features can help linear models capture non-linear patterns.

### Leakage Check
The target is not used.

### Final Decision
**Needs Further Analysis**

Polynomial features should be evaluated because they can increase model complexity and may lead to overfitting.

In [7]:
df["Age_Squared"] = df["Age"] ** 2

df[["Age", "Age_Squared"]].head()

,Age,Age_Squared
0,22.0,484.0
1,38.0,1444.0
2,26.0,676.0
3,35.0,1225.0
4,35.0,1225.0


## 6. Pairwise Interactions

Pairwise interactions combine pairs of features to capture relationships between them.

For example:

- Age × Pclass
- Age × Fare
- Fare × Pclass

However, creating every possible pair can produce a large number of features.

### Feature Name
`Fare_Pclass_Interaction`

### Source Columns
- `Fare`
- `Pclass`

### Logic
`Fare × Pclass`

### Reason
It represents the combined relationship between fare and passenger class.

### Business Meaning
In an e-commerce dataset, a similar interaction could be quantity × price.

### ML Relevance
It can help a model capture relationships between two numerical features.

### Leakage Check
The target variable is not used.

### Final Decision
**Needs Further Analysis**

The feature should be evaluated based on model performance and whether it provides information beyond the original features.

In [8]:
df["Fare_Pclass_Interaction"] = df["Fare"] * df["Pclass"]

df[["Fare", "Pclass", "Fare_Pclass_Interaction"]].head()

,Fare,Pclass,Fare_Pclass_Interaction
0,7.2500,3,21.7500
1,71.2833,1,71.2833
2,7.9250,3,23.7750
3,53.1000,1,53.1000
4,8.0500,3,24.1500


## 7. Domain-Based Interactions

Domain-based interactions are created using domain knowledge.

Instead of generating random combinations, we select features that have a logical relationship.

### Feature Name
`Family_Size`

### Source Columns
- `SibSp`
- `Parch`

### Logic
`SibSp + Parch + 1`

### Reason
The passenger's family group size may have a meaningful relationship with survival.

### Business Meaning
In an e-commerce dataset, domain knowledge could suggest creating:

`Total Spending / Number of Orders`

to represent average customer spending.

### ML Relevance
Domain-based features can provide meaningful information that generic mathematical combinations may not capture.

### Leakage Check
The target is not used.

### Final Decision
**Retain**

In [9]:
df["Family_Size"] = df["SibSp"] + df["Parch"] + 1

df[["SibSp", "Parch", "Family_Size"]].head()

,SibSp,Parch,Family_Size
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1


## Interaction Feature Comparison

| Technique | Example | Purpose |
|---|---|---|
| Feature × Feature | Age × Pclass | Capture combined effect |
| Feature + Feature | SibSp + Parch | Create a total |
| Feature − Feature | SibSp − Parch | Measure difference |
| Feature / Feature | Fare / Family Size | Create a ratio |
| Polynomial | Age² | Capture non-linear relationship |
| Pairwise Interaction | Fare × Pclass | Represent feature relationship |
| Domain-Based | Family Size | Use domain knowledge |

### Important Point

Interaction features should not be created simply because they are mathematically possible.

They should have a meaningful reason and should be evaluated for:

- Usefulness
- Redundancy
- Model performance
- Interpretability
- Complexity
- Leakage

In [10]:
interaction_features = [
    "Age_Pclass_Interaction",
    "Family_Total",
    "Sibling_Parent_Difference",
    "Fare_Per_Family_Member",
    "Age_Squared",
    "Fare_Pclass_Interaction",
    "Family_Size"
]

df[interaction_features].head()

,Age_Pclass_Interaction,Family_Total,Sibling_Parent_Difference,Fare_Per_Family_Member,Age_Squared,Fare_Pclass_Interaction,Family_Size
0,66.0,1,1,3.62500,484.0,21.7500,2
1,38.0,1,1,35.64165,1444.0,71.2833,2
2,78.0,0,0,7.92500,676.0,23.7750,1
3,35.0,1,1,26.55000,1225.0,53.1000,2
4,105.0,0,0,8.05000,1225.0,24.1500,1


## Required Feature Documentation

The important interaction features are documented below.

| Feature Name | Source Columns | Logic | Reason | Business Meaning | ML Relevance | Leakage Check | Final Decision |
|---|---|---|---|---|---|---|---|
| Age_Pclass_Interaction | Age, Pclass | Age × Pclass | Capture combined relationship | Combined age and class effect | Helps represent interactions | Target not used | Retain |
| Family_Total | SibSp, Parch | SibSp + Parch | Combine family relationship counts | Relatives traveling with passenger | Provides combined family information | Target not used | Retain |
| Sibling_Parent_Difference | SibSp, Parch | SibSp − Parch | Measure difference | Difference between relationship types | Captures relative relationship | Target not used | Retain |
| Fare_Per_Family_Member | Fare, SibSp, Parch | Fare / Family Size | Create relative fare | Fare associated with family member | Represents relative fare | Target not used | Retain |
| Age_Squared | Age | Age² | Capture non-linearity | Non-linear age effect | Helps linear models capture non-linear patterns | Target not used | Needs Further Analysis |
| Fare_Pclass_Interaction | Fare, Pclass | Fare × Pclass | Capture pairwise relationship | Combined fare and class effect | May improve predictive representation | Target not used | Needs Further Analysis |
| Family_Size | SibSp, Parch | SibSp + Parch + 1 | Represent complete group size | Total family group size | Useful group-level information | Target not used | Retain |

In [11]:
feature_documentation = pd.DataFrame({
    "Feature Name": [
        "Age_Pclass_Interaction",
        "Family_Total",
        "Sibling_Parent_Difference",
        "Fare_Per_Family_Member",
        "Age_Squared",
        "Fare_Pclass_Interaction",
        "Family_Size"
    ],
    "Source Columns": [
        "Age, Pclass",
        "SibSp, Parch",
        "SibSp, Parch",
        "Fare, SibSp, Parch",
        "Age",
        "Fare, Pclass",
        "SibSp, Parch"
    ],
    "Leakage Check": [
        "No direct target leakage",
        "No direct target leakage",
        "No direct target leakage",
        "No direct target leakage",
        "No direct target leakage",
        "No direct target leakage",
        "No direct target leakage"
    ],
    "Final Decision": [
        "Retain",
        "Retain",
        "Retain",
        "Retain",
        "Needs Further Analysis",
        "Needs Further Analysis",
        "Retain"
    ]
})

feature_documentation

,Feature Name,Source Columns,Leakage Check,Final Decision
0,Age_Pclass_Interaction,"Age, Pclass",No direct target leakage,Retain
1,Family_Total,"SibSp, Parch",No direct target leakage,Retain
2,Sibling_Parent_Difference,"SibSp, Parch",No direct target leakage,Retain
3,Fare_Per_Family_Member,"Fare, SibSp, Parch",No direct target leakage,Retain
4,Age_Squared,Age,No direct target leakage,Needs Further Analysis
5,Fare_Pclass_Interaction,"Fare, Pclass",No direct target leakage,Needs Further Analysis
6,Family_Size,"SibSp, Parch",No direct target leakage,Retain


## When Interaction Features Should Be Used

Interaction features are useful when:

- Two or more features have a meaningful relationship
- The effect of one feature depends on another
- Domain knowledge supports the interaction
- The interaction improves model performance
- The feature provides useful information not represented by the original features

### When Interaction Features Should Be Avoided

They should be avoided or carefully evaluated when:

- There is no logical relationship
- They create many unnecessary features
- They introduce redundancy
- They increase overfitting risk
- They make the model unnecessarily complex

### Final Conclusion

Interaction features can help machine learning models capture relationships between features that individual features cannot represent.

However, more features do not automatically mean a better model.

The best interaction features are those that have a clear reason, useful representation, acceptable complexity, and no data leakage.

In [12]:
df.shape, df[interaction_features].describe().T

((891, 19),
                            count         mean         std     min       25%  \
 Age_Pclass_Interaction     714.0    61.938151   34.379609  0.9200   38.0000   
 Family_Total               891.0     0.904602    1.613459  0.0000    0.0000   
 Sibling_Parent_Difference  891.0     0.141414    1.062212 -5.0000    0.0000   
 Fare_Per_Family_Member     891.0    19.916375   35.841257  0.0000    7.2500   
 Age_Squared                714.0  1092.761169  991.079188  0.1764  405.0625   
 Fare_Pclass_Interaction    891.0    51.543413   51.577796  0.0000   23.6874   
 Family_Size                891.0     1.904602    1.613459  1.0000    1.0000   
 
                              50%          75%        max  
 Age_Pclass_Interaction      58.0    81.000000   222.0000  
 Family_Total                 0.0     1.000000    10.0000  
 Sibling_Parent_Difference    0.0     0.000000     6.0000  
 Fare_Per_Family_Member       8.3    23.666667   512.3292  
 Age_Squared                784.0  1444.000000